# 01 · Veri Çekme
**Tez:** Güneş Döngüleri 23–25 Boyunca Güneş Patlamaları, Güneş Lekeleri ve Jeomanyetik Etkiler
**Hazırlayan:** Tunahan İnci · **Defter sürümü:** 1.2 (05.11.2026)

Bu defter tezde kullanılacak ham verileri resmi kaynaklarından indirir, okunur tablolara çevirir ve Google Drive'a kaydeder.

| # | Veri | Kaynak | Ne için |
|---|---|---|---|
| 1 | Güneş lekesi sayısı (günlük, aylık, 13 ay düzleştirilmiş) | SILSO, Belçika Kraliyet Gözlemevi | Güneş aktivitesi ana göstergesi |
| 2 | OMNI2 saatlik veri: Dst, Kp, ap, F10.7, güneş rüzgârı hızı, Bz | NASA SPDF | Dünya'ya etki (jeomanyetik fırtınalar) |
| 3 | GOES flare olay listesi | HEK (NOAA SWPC olayları) | Patlama sayısı ve şiddeti |
| 4 | GOES bilimsel kalite flare özetleri (keşif) | NOAA NCEI | Sürüm 2'de ana flare listesi olacak |

**Kurallar:** Ham veri asla elle düzenlenmez. Her indirme `data/raw` altına, her işlenmiş tablo `data/processed` altına yazılır. İndirilen her dosyanın adresi, tarihi ve SHA-256 özeti `data/indirme_kaydi.json` dosyasına işlenir.

**Çalıştırma:** Çalışma zamanı → Tümünü çalıştır. İlk çalıştırma 15–40 dk sürebilir (en uzun kısım flare listesi). Yarıda kalırsa tekrar çalıştır: inmiş dosyalar atlanır, kaldığı yerden devam eder.

In [ ]:
# 1) Google Drive'ı bağla ve klasörleri hazırla
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
KOK = Path('/content/drive/MyDrive/gunes_tezi')
HAM = KOK / 'data' / 'raw'
ISLENMIS = KOK / 'data' / 'processed'
SEKIL = KOK / 'figures'
for p in [HAM / 'silso', HAM / 'omni2', HAM / 'hek', HAM / 'ncei', ISLENMIS, SEKIL]:
    p.mkdir(parents=True, exist_ok=True)

BASLANGIC_YIL, BITIS_YIL = 2000, 2025
print('Proje klasörü:', KOK)

In [ ]:
# 2) Ortak araçlar: tekrar denemeli indirme + indirme kaydı
import hashlib, json, time, datetime as dt
import requests, numpy as np, pandas as pd

KAYIT = KOK / 'data' / 'indirme_kaydi.json'
_kayit = json.loads(KAYIT.read_text()) if KAYIT.exists() else {}

def indir(url, hedef, deneme=4, zorla=False):
    # Dosyayı indirir. Zaten varsa atlar. Kayda url, tarih, boyut ve sha256 yazar.
    hedef = Path(hedef)
    if hedef.exists() and not zorla and hedef.stat().st_size > 0:
        return hedef
    for i in range(deneme):
        try:
            r = requests.get(url, timeout=120)
            r.raise_for_status()
            hedef.write_bytes(r.content)
            _kayit[str(hedef.relative_to(KOK))] = {
                'url': url,
                'indirme_utc': dt.datetime.now(dt.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
                'bayt': len(r.content),
                'sha256': hashlib.sha256(r.content).hexdigest(),
            }
            KAYIT.write_text(json.dumps(_kayit, indent=2, ensure_ascii=False))
            return hedef
        except Exception as e:
            bekle = 5 * (i + 1)
            print(f'  ! {url} -> {e} | {bekle} sn sonra tekrar')
            time.sleep(bekle)
    raise RuntimeError(f'İndirilemedi: {url}')

print('Hazır. Kayıttaki dosya sayısı:', len(_kayit))

## 1 · Güneş lekesi sayısı (SILSO, Sürüm 2.0)
Dosyalar noktalı virgülle ayrılmış. Eksik değerler `-1` ile gösteriliyor, bunları `NaN` yapıyoruz. `kesin` sütunu 1 ise değer kesinleşmiş, 0 ise geçici.

In [ ]:
SILSO = 'https://www.sidc.be/SILSO/DATA/'
dosyalar = {
    'gunluk':  'SN_d_tot_V2.0.csv',
    'aylik':   'SN_m_tot_V2.0.csv',
    'duzlestirilmis': 'SN_ms_tot_V2.0.csv',
}
for ad, f in dosyalar.items():
    indir(SILSO + f, HAM / 'silso' / f)

g = pd.read_csv(HAM/'silso'/dosyalar['gunluk'], sep=';', header=None,
                names=['yil','ay','gun','ondalik','sn','std','n_gozlem','kesin'])
g['tarih'] = pd.to_datetime(dict(year=g.yil, month=g.ay, day=g.gun))
g.loc[g.sn < 0, 'sn'] = np.nan
g.loc[g['std'] < 0, 'std'] = np.nan
g = g[['tarih','sn','std','n_gozlem','kesin']]

def aylik_oku(f):
    d = pd.read_csv(HAM/'silso'/f, sep=';', header=None,
                    names=['yil','ay','ondalik','sn','std','n_gozlem','kesin'])
    d['tarih'] = pd.to_datetime(dict(year=d.yil, month=d.ay, day=1))
    d.loc[d.sn < 0, 'sn'] = np.nan
    d.loc[d['std'] < 0, 'std'] = np.nan
    return d[['tarih','sn','std','n_gozlem','kesin']]

a  = aylik_oku(dosyalar['aylik'])          # 1749'dan bugüne (tahmin modelleri için)
ad = aylik_oku(dosyalar['duzlestirilmis'])

g.to_parquet(ISLENMIS/'sn_gunluk.parquet', index=False)
a.to_parquet(ISLENMIS/'sn_aylik_1749.parquet', index=False)
ad.to_parquet(ISLENMIS/'sn_aylik_duzlestirilmis.parquet', index=False)

print('Günlük :', g.tarih.min().date(), '→', g.tarih.max().date(), f'({len(g):,} gün)')
print('Aylık  :', a.tarih.min().date(), '→', a.tarih.max().date())
# Döngü maksimumlarını 13 ay düzleştirilmiş seriden bul (kapsam çizelgesi için)
for bas, son, dongu in [('1999','2004','23'), ('2011','2016','24'), ('2022','2026','25')]:
    p = ad[(ad.tarih >= bas) & (ad.tarih <= son)].dropna()
    if len(p):
        r = p.loc[p.sn.idxmax()]
        print(f'Döngü {dongu} maksimumu: {r.tarih:%Y-%m}  düzleştirilmiş SN = {r.sn:.1f}')

## 2 · OMNI2 saatlik veri (Dünya tarafı)
Her yıl için bir dosya, 55 sütun, boşlukla ayrılmış. Kullandığımız sütunlar (1'den sayarak): 1 yıl, 2 yılın günü, 3 saat, 17 Bz (GSM, nT), 24 proton yoğunluğu, 25 güneş rüzgârı hızı (km/s), 39 Kp×10, 40 leke sayısı, 41 Dst (nT), 50 ap, 51 F10.7.

Eksik değer kodları sütuna göre değişiyor (ör. Dst için `99999`, Kp için `99`), aşağıda her biri ayrı temizleniyor.

**Fırtına eşikleri:** Kp ≥ 5 → G1 ve üstü (NOAA ölçeği). Dst ≤ −50 nT orta, ≤ −100 nT şiddetli fırtına.

In [ ]:
OMNI = 'https://spdf.gsfc.nasa.gov/pub/data/omni/low_res_omni/omni2_{y}.dat'
sutun = {0:'yil', 1:'gun', 2:'saat', 16:'bz_gsm', 23:'yogunluk', 24:'hiz',
         38:'kp10', 39:'r_omni', 40:'dst', 49:'ap', 50:'f107'}
eksik = {'bz_gsm':999.9, 'yogunluk':999.9, 'hiz':9999., 'kp10':99,
         'r_omni':999, 'dst':99999, 'ap':999, 'f107':999.9}

parcalar = []
for y in range(BASLANGIC_YIL, BITIS_YIL + 1):
    f = indir(OMNI.format(y=y), HAM/'omni2'/f'omni2_{y}.dat')
    d = pd.read_csv(f, sep=r'\s+', header=None, usecols=list(sutun)).rename(columns=sutun)
    parcalar.append(d)
    print(y, 'tamam', end=' | ')
o = pd.concat(parcalar, ignore_index=True)
for c, v in eksik.items():
    o.loc[np.isclose(o[c], v), c] = np.nan
o['zaman'] = (pd.to_datetime(o.yil.astype(str), format='%Y')
              + pd.to_timedelta(o.gun - 1, unit='D') + pd.to_timedelta(o.saat, unit='h'))
o['kp'] = o.kp10 / 10
o = o[['zaman','dst','kp','ap','f107','hiz','bz_gsm','yogunluk','r_omni']]
o.to_parquet(ISLENMIS/'omni2_saatlik.parquet', index=False)

gunluk = o.set_index('zaman').resample('D').agg(
    dst_min=('dst','min'), kp_max=('kp','max'), ap_ort=('ap','mean'),
    f107=('f107','mean'), hiz_max=('hiz','max'), bz_min=('bz_gsm','min')).reset_index()
gunluk['firtina_g1'] = gunluk.kp_max >= 5
gunluk['dst_sinif'] = pd.cut(gunluk.dst_min, [-np.inf,-250,-100,-50,np.inf],
                              labels=['süper','şiddetli','orta','sakin'])
gunluk.to_parquet(ISLENMIS/'omni2_gunluk.parquet', index=False)

print('\nSaatlik:', len(o), 'satır |', o.zaman.min(), '→', o.zaman.max())
print('Eksik oranları (%):'); print((o.isna().mean()*100).round(2).to_string())
print('\nEn güçlü 5 fırtına günü (Dst):')
print(gunluk.nsmallest(5, 'dst_min')[['zaman','dst_min','kp_max']].to_string(index=False))

## 3 · GOES flare olay listesi (HEK)
HEK, NOAA SWPC'nin GOES flare olaylarını sorgulanabilir şekilde sunar. Ay ay sorguluyoruz ve her ayı ayrı dosyaya yazıyoruz. Bağlantı koparsa tekrar çalıştırınca inmiş aylar atlanır.

⚠️ **Bilinen homojenlik sorunu:** SWPC'nin operasyonel sınıfları GOES 8–15 döneminde 0.7 ölçek çarpanı içeriyordu, GOES-16 ve sonrasında içermiyor. Yani aynı patlama eski dönemde yaklaşık %30 daha zayıf sınıflanmış görünür. Bu yüzden bu liste **ön analiz** içindir. Tezin ana flare listesi, ölçek düzeltmesi yapılmış NCEI bilimsel kalite flare özetleri olacak (Bölüm 4 ve 02 defteri). Bu farkın kendisi de tezde "veri homojenleştirme" başlığı altında anlatılacak.

In [ ]:
!pip -q install "sunpy[net]" "requests==2.32.4"

In [ ]:
from sunpy.net import Fido, attrs as sa
import warnings; warnings.filterwarnings('ignore')

ISTENEN = ['event_starttime','event_peaktime','event_endtime','fl_goescls',
           'ar_noaanum','hpc_x','hpc_y','frm_name']

def sutun_dizi(tbl, c):
    s = tbl[c]
    if hasattr(s, 'isot'):  return np.array(s.isot)
    if hasattr(s, 'value'): return np.array(s.value)
    return np.array(s)

def ay_sorgula(y, m):
    hedef = HAM/'hek'/f'hek_fl_{y}_{m:02d}.csv'
    if hedef.exists():
        return
    bas = pd.Timestamp(y, m, 1); son = bas + pd.offsets.MonthBegin(1)
    for i in range(4):
        try:
            r = Fido.search(sa.Time(bas.strftime('%Y-%m-%d'), son.strftime('%Y-%m-%d')),
                            sa.hek.EventType('FL'), sa.hek.OBS.Observatory == 'GOES')
            tbl = r['hek']
            d = pd.DataFrame({c: sutun_dizi(tbl, c) for c in ISTENEN if c in tbl.colnames})
            d.to_csv(hedef, index=False)
            return
        except Exception as e:
            print(f'  ! {y}-{m:02d}: {e}'); time.sleep(10*(i+1))
    print(f'  !! {y}-{m:02d} atlandı, sonra tekrar çalıştır')

for y in range(BASLANGIC_YIL, BITIS_YIL + 1):
    for m in range(1, 13):
        ay_sorgula(y, m)
    print(y, 'tamam', end=' | ')

In [ ]:
# Ayları birleştir, temizle, sınıfı akıya çevir
dosya = sorted((HAM/'hek').glob('hek_fl_*.csv'))
fl = pd.concat([pd.read_csv(f) for f in dosya if f.stat().st_size > 5], ignore_index=True)
for c in ['event_starttime','event_peaktime','event_endtime']:
    fl[c] = pd.to_datetime(fl[c], errors='coerce')
fl = fl.dropna(subset=['event_peaktime','fl_goescls'])
fl['fl_goescls'] = fl.fl_goescls.astype(str).str.strip().str.upper()
fl = fl[fl.fl_goescls.str.match(r'^[ABCMX]\d')]

taban = {'A':1e-8, 'B':1e-7, 'C':1e-6, 'M':1e-5, 'X':1e-4}
fl['sinif'] = fl.fl_goescls.str[0]
fl['pik_aki'] = fl.sinif.map(taban) * pd.to_numeric(fl.fl_goescls.str[1:], errors='coerce')

once = len(fl)
fl = fl.sort_values('event_peaktime').drop_duplicates(subset=['event_peaktime','fl_goescls'])
print(f'Tekrar eden {once-len(fl)} kayıt silindi.')
fl.to_parquet(ISLENMIS/'flare_hek.parquet', index=False)

print('Toplam flare:', len(fl), '|', fl.event_peaktime.min(), '→', fl.event_peaktime.max())
print(fl.sinif.value_counts().reindex(list('ABCMX')).fillna(0).astype(int).to_string())
print('\nEn güçlü 10 patlama:')
print(fl.nlargest(10, 'pik_aki')[['event_peaktime','fl_goescls','ar_noaanum']].to_string(index=False))

## 4 · NCEI bilimsel kalite flare özetleri (keşif)
NOAA NCEI, GOES XRS verisini yeniden kalibre edip bilimsel kalite ürünler yayımlıyor (GOES-R serisi için Şubat 2026'da sürüm 2-2-1). Bu hücre yalnızca ilgili klasörleri tarar ve bulduğu flare özeti (`flsum`) dosyalarını listeler. Hiçbir şey indirmez. Çıktıya göre 02 defterinde indirme ve birleştirme yazılacak.

In [ ]:
import re
from urllib.parse import urljoin

def linkler(url):
    try:
        r = requests.get(url, timeout=60); r.raise_for_status()
        return [urljoin(url, h) for h in re.findall(r'href="([^"?#]+)"', r.text)
                if not h.startswith(('/', 'http')) or h.startswith(url)]
    except Exception as e:
        print('  erişilemedi:', url, '|', e); return []

def flsum_ara(kok, derinlik=3):
    bulunan, kuyruk = [], [(kok, 0)]
    while kuyruk:
        u, d = kuyruk.pop(0)
        for l in linkler(u):
            if 'flsum' in l.lower() and l.endswith('.nc'):
                bulunan.append(l)
            elif l.endswith('/') and d < derinlik and l.startswith(kok) and l != u:
                if 'flsum' in l.lower() or d < 1:
                    kuyruk.append((l, d+1))
    return bulunan

kokler = [f'https://data.ngdc.noaa.gov/platforms/solar-space-observing-satellites/goes/goes{n}/l2/data/'
          for n in (16, 17, 18, 19)]
kokler += ['https://www.ncei.noaa.gov/data/goes-space-environment-monitor/access/science/xrs/']

ncei = {}
for k in kokler:
    print('Taranıyor:', k)
    ncei[k] = flsum_ara(k)
    for f in ncei[k][:5]: print('   ', f)
    if len(ncei[k]) > 5: print(f'    ... toplam {len(ncei[k])} dosya')
(HAM/'ncei'/'flsum_bulunanlar.json').write_text(json.dumps(ncei, indent=2))
print('\nListe kaydedildi: data/raw/ncei/flsum_bulunanlar.json')

## 5 · Özet ve ilk bakış grafiği
Üç veri setini aylık olarak aynı eksende gösterir: leke sayısı, M ve X sınıfı patlama sayısı, aylık en düşük Dst. Bu tezin ana hikâyesinin ilk taslağı.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

pencere = (pd.Timestamp(f'{BASLANGIC_YIL}-01-01'), pd.Timestamp(f'{BITIS_YIL}-12-31'))
sn_ay = a[(a.tarih >= pencere[0]) & (a.tarih <= pencere[1])].set_index('tarih')
sn_dz = ad[(ad.tarih >= pencere[0]) & (ad.tarih <= pencere[1])].set_index('tarih')
buyuk = fl[fl.sinif.isin(['M','X'])].set_index('event_peaktime').resample('MS').size()
x_ay = fl[fl.sinif == 'X'].set_index('event_peaktime').resample('MS').size()
dst_ay = gunluk.set_index('zaman').dst_min.resample('MS').min()

fig, ax = plt.subplots(3, 1, figsize=(13, 9), sharex=True, constrained_layout=True)
ax[0].bar(sn_ay.index, sn_ay.sn, width=25, color='#f5a524', alpha=.45, label='Aylık')
ax[0].plot(sn_dz.index, sn_dz.sn, color='#b86e00', lw=2, label='13 ay düzleştirilmiş')
ax[0].set_ylabel('Güneş lekesi sayısı'); ax[0].legend(loc='upper right')
ax[1].bar(buyuk.index, buyuk.values, width=25, color='#3fa0c9', label='M + X')
ax[1].bar(x_ay.index, x_ay.values, width=25, color='#c4432c', label='X')
ax[1].set_ylabel('Patlama / ay'); ax[1].legend(loc='upper right')
ax[2].bar(dst_ay.index, dst_ay.values, width=25, color='#3451c4')
for e, ad_ in [(-50,'orta'), (-100,'şiddetli'), (-250,'süper')]:
    ax[2].axhline(e, color='grey', lw=.8, ls='--'); ax[2].text(pencere[0], e, f' {ad_}', va='bottom', fontsize=8, color='grey')
ax[2].set_ylabel('Aylık en düşük Dst (nT)')
ax[2].xaxis.set_major_locator(mdates.YearLocator(2)); ax[2].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
for x in ax: x.grid(alpha=.25)
fig.suptitle("Güneş aktivitesi ve Dünya'ya etkisi, 2000–2025 (ön bakış, HEK flare listesi)", fontsize=13)
fig.savefig(SEKIL/'01_ilk_bakis.png', dpi=200)
plt.show()

ozet = pd.DataFrame([
    ['SILSO günlük SN', len(g), g.tarih.min(), g.tarih.max(), g.sn.isna().mean()*100],
    ['OMNI2 saatlik',  len(o), o.zaman.min(), o.zaman.max(), o.dst.isna().mean()*100],
    ['HEK flare',      len(fl), fl.event_peaktime.min(), fl.event_peaktime.max(), np.nan],
], columns=['veri','satır','başlangıç','bitiş','eksik_%'])
ozet.to_csv(ISLENMIS/'01_veri_ozeti.csv', index=False)
ozet